<center style="padding:1rem 0;">
    <h1 style="font-size: 4rem;">IA - Deep Learning</h1>
    <h2 style="font-size: 2rem;">Livrable 2 - Construction d'un premier réseau de neurones</h2>
    <h5 style="font-size: 1rem;"><i>Thomas VINET, Hugo HELM, Alban GODIER</i></h5>
</center>

<img src="assets/cesi.png" style="position:absolute;right:2rem;top:4.5rem;width:10rem;background:#fee237;"/>

In [1]:
from typing import Optional, Dict, Tuple, List

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

from lib.neural_network import NeuralNetwork, DrawRealTimeLoss, EarlyStopping, Layer, Evaluation
from lib.neural_network.grid_search import GridSearch

warnings.filterwarnings("ignore")
np.random.seed(42)

In [2]:
# Chargement des données
df_train = pd.read_csv('dataset/dataset_train.csv')
df_validation = pd.read_csv('dataset/dataset_validation.csv')

# Pour tester, on peut réduire la taille des données d'entraînement
# df_train = df_train.sample(n=10000, random_state=42).reset_index(drop=True)
# df_validation = df_validation.sample(n=2000, random_state=42).reset_index(drop=True)

print(f"Données d'entraînement: {df_train.shape}")
print(f"Données de validation: {df_validation.shape}")
print(f"\nPremières lignes des données d'entraînement:")
print(df_train.head())
print(f"\nInformations sur les données:")
print(df_train.info())

Données d'entraînement: (59582, 17)
Données de validation: (22384, 17)

Premières lignes des données d'entraînement:
   Diabetes_binary  HighBP  HighChol  CholCheck      BMI  Smoker  Stroke  \
0              0.0     0.0       0.0        1.0  0.31250     0.0     0.0   
1              0.0     1.0       0.0        1.0  0.34375     1.0     0.0   
2              1.0     1.0       1.0        1.0  0.75000     0.0     0.0   
3              1.0     1.0       0.0        1.0  0.53125     0.0     0.0   
4              0.0     1.0       1.0        1.0  0.43750     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  HvyAlcoholConsump  \
0                   0.0           1.0     1.0      1.0                0.0   
1                   0.0           1.0     0.0      1.0                0.0   
2                   1.0           0.0     1.0      1.0                0.0   
3                   0.0           0.0     0.0      1.0                0.0   
4                   0.0           0.0    

In [3]:
# Séparation des features et de la cible
# La colonne cible est 'Diabetes_binary'
target_column = 'Diabetes_binary'

X_train = df_train.drop(columns=[target_column]).astype(float)
y_train = df_train[target_column].astype(int)

X_validation = df_validation.drop(columns=[target_column]).astype(float)
y_validation = df_validation[target_column].astype(int)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_validation shape: {X_validation.shape}, y_validation shape: {y_validation.shape}")
print(f"\nDistribution de y_train: {np.bincount(y_train)}")
print(f"Distribution de y_validation: {np.bincount(y_validation)}")

X_train shape: (59582, 16), y_train shape: (59582,)
X_validation shape: (22384, 16), y_validation shape: (22384,)

Distribution de y_train: [29791 29791]
Distribution de y_validation: [19074  3310]


In [4]:
# Définition des architectures et paramètres pour la grid search
from lib.neural_network.activation.relu import Relu
from lib.neural_network.activation.sigmoid import Sigmoid
from lib.neural_network.activation.none import NoActivation
from lib.neural_network.loss.mean_squared_error import MeanSquaredError
from lib.neural_network.loss.binary_cross_entropy import BinaryCrossEntropy
from lib.neural_network.loss.binary_cross_entropy_sigmoid import BinaryCrossEntropySigmoid
from lib.neural_network.grid_search import GridSearch, Params
from lib.neural_network.callback.draw_real_time_loss import DrawRealTimeLoss
from lib.neural_network.callback.progress_bar import ProgressBar
from lib.neural_network.callback.carbon_emissions import CarbonEmissions

nn = NeuralNetwork([
    Layer(neurons=64, activation=Relu(), dropout_rate=0.2),
    Layer(neurons=32, activation=Relu(), dropout_rate=0.2),
    Layer(neurons=1, activation=Sigmoid(), dropout_rate=0.0),
], loss=MeanSquaredError(), inputs=X_train.shape[1])
carbon = CarbonEmissions()
nn.add_callback(EarlyStopping())
nn.add_callback(carbon)
# nn.add_callback(DrawRealTimeLoss())
# nn.add_callback(ProgressBar())

nn.fit(
    X_train.to_numpy(),
    y_train.to_numpy(),
    epochs=100,
    batch_size=500,
    learning_rate=0.01
)

carbon.draw_result()

[codecarbon WARNING @ 12:00:24] Multiple instances of codecarbon are allowed to run at the same time.


,timestamp,duration,emissions,emissions_rate,energy_consumed
0,2026-04-16T12:00:31,2.483978,9.851373e-07,3.965966e-07,0.000018


In [ ]:
from lib.neural_network.explainatinator.lime import LIME

explainer = LIME(nn)

patient = X_validation.to_numpy()[0].reshape(1, -1)
print("Patient features:")
print(patient)
explanation = explainer.explain(patient)
print("Explanation:")
print(explanation)